# Environment Check

In [1]:
# Verify GPU and environment
import subprocess, platform, torch

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


# Install Dependencies

In [2]:
!pip install ultralytics onnx onnxruntime onnxslim -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 66.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1,

# Download DeepPCB Dataset

In [3]:
!git clone https://github.com/tangsanli5201/DeepPCB.git

Cloning into 'DeepPCB'...
remote: Enumerating objects: 4753, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4753 (delta 13), reused 12 (delta 12), pack-reused 4739 (from 1)
Receiving objects: 100% (4753/4753), 85.81 MiB | 38.42 MiB/s, done.
Resolving deltas: 100% (191/191), done.


In [4]:
import os

base = "./DeepPCB/PCBData"
groups = os.listdir(base)
print(f"Groups found: {len(groups)}")
print(groups[:5])

Groups found: 13
['test.txt', 'group12100', 'group13000', 'group90100', 'group20085']


# Convert DeepPCB Annotations to YOLO format

In [7]:
import os, shutil, random
from pathlib import Path
from PIL import Image

DEFECT_CLASSES = {
    '1': 0,  # open
    '2': 1,  # short
    '3': 2,  # mousebite
    '4': 3,  # spur
    '5': 4,  # copper
    '6': 5,  # pin-hole
}

def convert_annotation(ann_path, img_w, img_h):
    lines = []
    with open(ann_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            x1, y1, x2, y2, cls = parts[0], parts[1], parts[2], parts[3], parts[4]
            if cls not in DEFECT_CLASSES:
                continue
            cx = (int(x1) + int(x2)) / 2 / img_w
            cy = (int(y1) + int(y2)) / 2 / img_h
            w  = (int(x2) - int(x1)) / img_w
            h  = (int(y2) - int(y1)) / img_h
            lines.append(f"{DEFECT_CLASSES[cls]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines

for split in ['train', 'val']:
    Path(f"dataset/{split}/images").mkdir(parents=True, exist_ok=True)
    Path(f"dataset/{split}/labels").mkdir(parents=True, exist_ok=True)

all_samples = []
base = "./DeepPCB/PCBData"

for group in sorted(os.listdir(base)):
    group_path = os.path.join(base, group)
    img_dir = os.path.join(group_path, group.replace("group", ""))   
    ann_dir = os.path.join(group_path, group.replace("group", "") + "_not")  

    if not os.path.isdir(img_dir) or not os.path.isdir(ann_dir):
        continue

    for fname in os.listdir(img_dir):
        if "_test.jpg" not in fname:
            continue
        img_path = os.path.join(img_dir, fname)
        # annotation name: strip _test.jpg to .txt
        ann_name = fname.replace("_test.jpg", ".txt")
        ann_path = os.path.join(ann_dir, ann_name)
        if os.path.exists(ann_path):
            all_samples.append((img_path, ann_path))

random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.85)
splits = {'train': all_samples[:split_idx], 'val': all_samples[split_idx:]}

for split, samples in splits.items():
    for img_path, ann_path in samples:
        img = Image.open(img_path)
        w, h = img.size
        label_lines = convert_annotation(ann_path, w, h)
        fname = os.path.basename(img_path)
        shutil.copy(img_path, f"dataset/{split}/images/{fname}")
        with open(f"dataset/{split}/labels/{fname.replace('_test.jpg', '.txt')}", 'w') as f:
            f.write("\n".join(label_lines))

print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])}")

Train: 1275 | Val: 225


In [11]:
import os
from pathlib import Path

for split in ['train', 'val']:
    img_dir = Path(f"dataset/{split}/images")
    for img in img_dir.glob("*_test.jpg"):
        new_name = img.stem.replace("_test", "") + ".jpg"
        img.rename(img_dir / new_name)

print("Images renamed. Verifying...")

# Verify alignment
img_dir = Path("dataset/train/images")
lbl_dir = Path("dataset/train/labels")
imgs = sorted(img_dir.glob("*.jpg"))[:5]
for img in imgs:
    expected_label = lbl_dir / (img.stem + ".txt")
    print(f"Image: {img.name} | Label exists: {expected_label.exists()}")

Images renamed. Verifying...
Image: 00041000.jpg | Label exists: True
Image: 00041001.jpg | Label exists: True
Image: 00041002.jpg | Label exists: True
Image: 00041003.jpg | Label exists: True
Image: 00041005.jpg | Label exists: True


# YAML Config

In [8]:
yaml_content = """
path: /kaggle/working/dataset
train: train/images
val: val/images

nc: 6
names: ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']
"""

with open("deeppcb.yaml", "w") as f:
    f.write(yaml_content)

print("YAML written.")

YAML written.


# Train

In [12]:
from ultralytics import YOLO
import shutil, os

model = YOLO("yolo26n.pt")

results = model.train(
    data="deeppcb.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/kaggle/working/runs", 
    name="qchip",
    patience=10,
    save=True,
    save_period=5,  
)

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=deeppcb.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=qchip-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0, p

# Evaluate

In [13]:
metrics = model.val(data="deeppcb.yaml", device=0)
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 730.9±164.0 MB/s, size: 32.8 KB)
val: Scanning /kaggle/working/dataset/val/labels.cache... 225 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 225/225 94.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.4it/s 3.4s0.1s
                   all        225       1470      0.913      0.888      0.957      0.731
                  open        202        289      0.933      0.914      0.973      0.673
                 short        172        222      0.852      0.855      0.935       0.65
             mousebite        188        284      0.923      0.923      0.969      0.719
                  spur        174        236      0.954      0.864      0.956       0.71
                copper        179     

# Export to ONNX FP32

In [16]:
model.export(
    format="onnx",
    imgsz=640,
    opset=17,
    simplify=True, # onnxslim cleans the graph
    dynamic=False, # fixed batch size = faster CPU inference
)

# The exported file lands next to the best.pt
import glob
onnx_fp32 = glob.glob("/kaggle/working/runs/qchip-2/weights/best.onnx")[0]
print(f"Exported: {onnx_fp32}")
print(f"Size: {os.path.getsize(onnx_fp32)/1e6:.1f} MB")

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)

PyTorch: starting from '/kaggle/working/runs/qchip-2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.21.0 opset 17...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 1.6s, saved as '/kaggle/working/runs/qchip-2/weights/best.onnx' (9.4 MB)

Export complete (1.8s)
Results saved to /kaggle/working/runs/qchip-2/weights/best.onnx
Predict:         yolo predict task=detect model=/kaggle/working/runs/qchip-2/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/qchip-2/weights/best.onnx imgsz=640 data=deeppcb.yaml  
Visualize:       https://netron.app
Exported: /kaggle/working/runs/qchip-2/weights/best.onnx
Size: 9.8 MB


# INT8 Post-Training Quantization

In [19]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType
import onnxruntime as ort
import numpy as np
from PIL import Image

cal_images = list(Path("dataset/val/images").glob("*.jpg"))[:150]

class PCBCalibrationReader(CalibrationDataReader):
    def __init__(self, image_paths, input_name):
        self.paths = iter(image_paths)
        self.input_name = input_name

    def get_next(self):
        path = next(self.paths, None)
        if path is None:
            return None
        img = Image.open(path).convert("RGB").resize((640, 640))
        arr = np.array(img).transpose(2, 0, 1).astype(np.float32) / 255.0
        return {self.input_name: arr[np.newaxis, :]}

sess = ort.InferenceSession(onnx_fp32)
input_name = sess.get_inputs()[0].name
onnx_int8 = onnx_fp32.replace(".onnx", "_int8.onnx")

quantize_static(
    model_input=onnx_fp32,
    model_output=onnx_int8,
    calibration_data_reader=PCBCalibrationReader(cal_images, input_name),
    weight_type=QuantType.QInt8,  # fixed
)

print(f"INT8 model: {onnx_int8}")
print(f"FP32 size: {os.path.getsize(onnx_fp32)/1e6:.1f} MB")
print(f"INT8 size: {os.path.getsize(onnx_int8)/1e6:.1f} MB")

INT8 model: /kaggle/working/runs/qchip-2/weights/best_int8.onnx
FP32 size: 9.8 MB
INT8 size: 3.0 MB


# Save for download

In [20]:
import shutil

shutil.copy(onnx_fp32, "/kaggle/working/qchip_fp32.onnx")
shutil.copy(onnx_int8, "/kaggle/working/qchip_int8.onnx")

# Also save val images for local testing
shutil.copytree("dataset/val/images", "/kaggle/working/val_images", dirs_exist_ok=True)

print("Files ready in /kaggle/working/")

Files ready in /kaggle/working/


In [21]:
import shutil

shutil.make_archive("/kaggle/working/val_images", "zip", "/kaggle/working/val_images")
print("Done. Download val_images.zip from the Output panel")

Done. Download val_images.zip from the Output panel
